In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path


In [3]:
Path.cwd()

PosixPath('/workspaces/GroundwaterData/notebooks')

In [26]:
import csv


nwdp_data_path = Path("..") / "downloaded_data"

processed_data_path = Path("..") / "data"
all_locations = set()

for file in nwdp_data_path.rglob("*.csv"):
    try:
        with open(file, "r", encoding="latin-1") as f:
            row_count = sum(1 for _ in f) - 1
        print(f"{file}: {row_count}")
        if row_count > 0:


            with open(file, encoding="latin-1", newline="") as f:
                reader = csv.DictReader(f)

                for row in reader:
                    try:
                        all_locations.add(
                            (
                                float(row["Latitude"]),
                                float(row["Longitude"])
                            )
                        )
                    except ValueError:
                        continue

    
    except UnicodeDecodeError as e:
        print(f"FAILED: {file}")
        print(e)

print(f"{len(all_locations):,} unique coordinates")

../downloaded_data/nwdp/odisha/gwl_tel_6_hourly_cgwb_od_2026_2030.csv: 58936
../downloaded_data/nwdp/odisha/gwl_manual_quarterly_odisha-gw_od_1950_1990.csv: 3
../downloaded_data/nwdp/odisha/gwl_tel_6_hourly_odisha_gw_od_2026_2030.csv: 531246


../downloaded_data/nwdp/odisha/gwl_tel_6_hourly_cgwb_od_2021_2025.csv: 220221
../downloaded_data/nwdp/odisha/gwl_manual_quarterly_odisha_gw_od_2026_2030.csv: 0
../downloaded_data/nwdp/odisha/gwq_physical_parameter_manual_cgwb_od_1961_2025.csv: 12728
../downloaded_data/nwdp/odisha/gwl_manual_quarterly_odisha_gw_od_2021_2025.csv: 4658
../downloaded_data/nwdp/odisha/gwl_manual_quarterly_odisha-gw_od_1991_2020.csv: 160109
../downloaded_data/nwdp/odisha/gwq_chemical_parameter_manual_cgwb_od_1961_2025.csv: 15400
../downloaded_data/nwdp/odisha/gwl_tel_6_hourly_odisha-gw_od_1991_2020.csv: 822
../downloaded_data/nwdp/odisha/gwl_manual_quarterly_cgwb_od_2021_2025.csv: 16379
../downloaded_data/nwdp/odisha/gwl_manual_quarterly_cgwb_od_2026_2030.csv: 0
../downloaded_data/nwdp/odisha/gwq_physical_parameter_manual_cgwb_od_2026_2030.csv: 0
../downloaded_data/nwdp/odisha/gwq_chemical_parameter_manual_cgwb_od_2026_2030.csv: 0
../downloaded_data/nwdp/odisha/gwl_tel_6_hourly_odisha_gw_od_2021_2025.csv: 33

In [27]:
with open(processed_data_path / "unique_locations.csv", "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow(["Latitude", "Longitude"])

    for lat, lon in sorted(all_locations):
        writer.writerow([lat, lon])

# Estiamte area

In [29]:
import geopandas as gpd

polygons_path = processed_data_path / "GWR"

for polygon in polygons_path.rglob("*.geojson"):
    gdf = gpd.read_file(polygon)

    # Reproject to a metric CRS
    gdf_m = gdf.to_crs("EPSG:3857")

    # area in m²
    gdf["area_m2"] = gdf_m.area

    # area in km²
    gdf["area_km2"] = gdf["area_m2"] / 1_000_000

    print(gdf[["area_km2"]])

      area_km2
0  8198.781068
       area_km2
0  59755.160941


In [31]:
from shapely.geometry import Point

points_gdf = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for lat, lon in all_locations],
    crs="EPSG:4326").to_crs(3857)

In [30]:
area_jharkhand = gpd.read_file(polygons_path / "study_area_jharkhand.geojson")
area_odisha = gpd.read_file(polygons_path / "study_area_odisha.geojson")

area_jharkhand = area_jharkhand.to_crs(3857)
area_odisha = area_odisha.to_crs(3857)

area_jharkhand["geometry"] = area_jharkhand.buffer(10_000)  # 10 km
area_odisha["geometry"] = area_odisha.buffer(10_000)

In [32]:
hits_jharkhand = gpd.sjoin(points_gdf, area_jharkhand, predicate="within")
hits_odisha = gpd.sjoin(points_gdf, area_odisha, predicate="within")

In [34]:
hits_odisha

,geometry,index_right,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,id2
3,POINT (9125353.413 2070022.577),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
6,POINT (9211749.708 2130031.374),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
8,POINT (9208533.811 2132742.549),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
27,POINT (9192206.952 2136954.425),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
115,POINT (9187877.861 2150030.954),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4499,POINT (9227139.008 2112620.749),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
4508,POINT (9218177.789 2108873.423),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
4545,POINT (9186331.757 2139080.96),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0
4576,POINT (9169417.379 2159645.833),0,study_area_odisha.1,None,None,None,None,None,None,-1,0,-1,None,None,1.0


In [35]:
jharkhand_points = {
    (round(g.y, 6), round(g.x, 6))
    for g in hits_jharkhand.to_crs(4326).geometry
}

odisha_points = {
    (round(g.y, 6), round(g.x, 6))
    for g in hits_odisha.to_crs(4326).geometry
}

In [36]:

# Save Jharkhand points
with open(processed_data_path / "jharkhand_locations.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Latitude", "Longitude"])

    for lat, lon in sorted(jharkhand_points):
        writer.writerow([lat, lon])

# Save Odisha points
with open(processed_data_path / "odisha_locations.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Latitude", "Longitude"])

    for lat, lon in sorted(odisha_points):
        writer.writerow([lat, lon])

In [15]:
import chardet

file = "../downloaded_data/nwdp/jharkhand/temprature_tele_hr_jharkhand_jh_1961_2020.csv"

encodings = [
    "utf-8",
    "utf-8-sig",
    "cp1252",
    "latin-1",
    "utf-16"
]

for enc in encodings:
    try:
        with open(file, encoding=enc) as f:
            f.readline()
        print(f"Works: {enc}")
    except UnicodeDecodeError:
        pass

Works: cp1252
Works: latin-1


In [6]:
df

,Agency,Station,State,District,Latitude,Longitude,Category,Data Type,Shared Location
0,CGWB,1Bantala,Odisha,ANUGUL,20.750556,85.065000,groundwater_level,manual_quarterly,False
1,CGWB,AG Office,Jharkhand,RANCHI,23.341111,85.315833,groundwater_level,manual_quarterly,False
2,CGWB,AMD Campus,Jharkhand,EAST SINGHBHUM (Jamshedpur),22.756383,86.201289,groundwater_level,manual_quarterly,True
3,CGWB,AMD Campus,Jharkhand,EAST SINGHBHUM (Jamshedpur),22.756383,86.201289,groundwater_level,telemetry_6_hourly,True
4,CGWB,Abhimanpur,Odisha,CUTTACK,20.444444,85.437500,groundwater_level,manual_quarterly,False
...,...,...,...,...,...,...,...,...,...
9600,Odisha GW,"Uttareswar High School, sona , Balasore",Odisha,Baleshwar,21.399306,86.803046,groundwater_level,telemetry_6_hourly,False
9601,Odisha GW,VCC Playground near shyam sunder Mandira,Odisha,JAJAPUR,20.644169,86.143998,groundwater_level,telemetry_6_hourly,False
9602,Odisha GW,Vibekananda Vidyapatha,Odisha,CUTTACK,20.483928,86.018116,groundwater_level,telemetry_6_hourly,False
9603,Odisha GW,Water Shed Reasearch Center Cum Training Hall,Odisha,JHARSUGUDA,21.858500,84.044139,groundwater_level,telemetry_6_hourly,False


In [8]:
df["Data Type"].unique()  

<ArrowStringArray>
[  'manual_quarterly', 'telemetry_6_hourly',             'manual',
   'telemetry_hourly',       'manual_daily']
Length: 5, dtype: str

In [15]:

# Read the CSV; assume coordinates are longitude and latitude
df = pd.read_csv(file)
df[["Longitude", "Latitude"]].drop_duplicates().values

array([[85.7708333 , 20.317222  ],
       [85.72777778, 20.24888889],
       [85.8275    , 20.30416667],
       [85.02128   , 20.376704  ],
       [85.83666667, 20.90972222],
       [85.5933    , 20.9139    ],
       [86.1875    , 21.2086    ],
       [86.22361111, 20.7475    ],
       [86.00416667, 20.85972222],
       [86.18583333, 19.95361111],
       [85.06555556, 20.74166667],
       [86.9336    , 21.3897    ],
       [85.65      , 21.5       ],
       [83.76      , 21.74      ],
       [85.8367    , 20.8842    ],
       [85.82361111, 19.95916667],
       [84.6508    , 20.9689    ],
       [86.062531  , 20.814473  ],
       [85.2192    , 20.8867    ],
       [82.75694444, 20.29638889],
       [84.23968889, 19.19597778],
       [85.63277778, 19.85444444],
       [86.07916667, 20.9625    ],
       [85.233617  , 20.075519  ],
       [86.07805556, 20.7125    ],
       [83.4111502 , 20.5356189 ],
       [85.03555556, 20.83833333],
       [82.35250833, 19.66834722],
       [85.715     ,

In [ ]:

points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
)

# Read the GeoJSON polygon
polygon = gpd.read_file("polygon.geojson")

# Make sure both layers use the same coordinate reference system
points = points.to_crs(polygon.crs)

# If the GeoJSON contains multiple features, treat them as one area
area = polygon.geometry.union_all()

# `within` excludes points exactly on the boundary
df["inside"] = points.geometry.within(area)

# Save the results, including the original CSV columns
df.to_csv("points_checked.csv", index=False)